[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_routing.ipynb)


# <center>OpenAI 智能体模式：路由</center>

本入门指南介绍如何使用 `openai-agents` 库构建智能体循环。

这一模式通过路由为特定子任务选择专业智能体，由其尝试完成子任务并返回结果。

下面构建一个根据用户投资策略创建股票和 ETF 投资组合的智能体。
1. **路由智能体**：根据用户投资策略选择工作智能体。
2. **研究智能体**：在网上搜索能够支持用户投资策略的股票和 ETF 信息。
3. **问答智能体**：回答有关沃伦·巴菲特式投资的问题。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建智能体

In [ ]:
from pprint import pprint
from textwrap import dedent

from agents import Agent, Runner, TResponseInputItem, WebSearchTool
from agents.model_settings import ModelSettings

qa_agent = Agent(
    name="Investing Q&A Agent",
    instructions=dedent("""你是沃伦·巴菲特，正在回答投资相关问题。"""),
    model="gpt-4.1",
)

research_agent = Agent(
    name="Financial Search Agent",
    instructions=dedent(
        """你是专注金融主题的研究助手。给定股票代码后，使用网络搜索获取最新背景，生成不超过 50 个词的简短摘要，重点关注对金融分析师有用的关键数字、事件或引述。"""
    ),
    model="gpt-4.1",
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required", parallel_tool_calls=True),
)

orchestrator_agent = Agent(
    name="Routing Agent",
    instructions=dedent(
        """你是资深金融分析师，任务是转交给适当的智能体或工具。"""
    ),
    model="gpt-4.1",
    handoffs=[
        research_agent,
        qa_agent,
    ],
)

In [ ]:
input_items: list[TResponseInputItem] = []

while True:
    user_input = input("请输入问题： ")
    if user_input == "exit":
        break
    input_item = {"content": user_input, "role": "user"}
    input_items.append(input_item)
    orchestrator = await Runner.run(orchestrator_agent, input_items)
    orchestrator_output = orchestrator.final_output
    pprint(orchestrator.last_agent)
    pprint(orchestrator_output)
    input_items = orchestrator.to_input_list()